# M.M.M Make Mincraft Mode

- **Full**: 플랜 생성 후 제작
- **Plan**: 플랜만 생성
- **Revise**: 기존 모드 ZIP을 수정
- **Execute**: 저장된 플랜으로 바로 제작
- **Audit**: 프로젝트 전체 진단
- **Debug Mode**: 기존 Fast Mode를 대체합니다. `DEBUG_MODE=True`이면 LLM planner/research/plan generation을 호출하지 않고 검증된 예제 플랜을 주입한 뒤 일반 제작/검증 경로로 바로 들어갑니다.


In [ ]:
# @title 1. 실행 모드 및 설정
import os

RUN_MODE = "Full" #@param ["Full", "Plan", "Revise", "Execute", "Audit"]
PROMPT = "계절마다 다른 작물을 재배하고 요리하는 모드를 만들어줘." #@param {type:"string"}
PLAN_FILE = "" #@param {type:"string"}
MINECRAFT_VERSION = "Auto" #@param ["Auto", "26.2", "26.1.1", "26.1", "1.21.10", "1.21.8", "1.21.5", "1.21.4", "1.21.1", "1.20.6", "1.20.4", "1.20.1"]
MOD_LOADER = "Auto" #@param {type:"string"}
REFERENCE_MOD_URLS = "" #@param {type:"string"}
MODEL_PROFILE = "Qwen3.5-9B_6GB" #@param ["Qwen3.5-9B_6GB", "Qwen3.6-35B_23GB", "Qwen3.8-27B_18GB", "fast_test", "remote_quality"]
PERFORMANCE_MODE = "Auto" #@param ["Auto", "Latency", "Throughput"]
KV_CACHE_QUANT = "q4_0" #@param ["q4_0", "q8_0", "f16"]
KV_CACHE_AUTOTUNE = True #@param {type:"boolean"}
DEBUG_MODE = False #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
ALLOW_REMOTE_TRAJECTORY_STORE = False #@param {type:"boolean"}
CURSEFORGE_API_KEY = "" #@param {type:"string"}

REMOTE_BASE_URL = REMOTE_TEXT_MODEL = REMOTE_IMAGE_MODEL = REMOTE_SPEECH_MODEL = ""
RUN_BLOCKBENCH = RUN_RUNTIME = RUN_CLIENT = RUN_MINEFLAYER = RUN_VISUAL_REVIEW = False
ACCEPT_EULA = False
SERVER_LAUNCHER = ""
RUN_NAME = "complete-colab-run"
SCREENSHOTS = []

VALID_RUN_MODES = {"Full", "Plan", "Revise", "Execute", "Audit"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"지원하지 않는 실행 모드: {RUN_MODE}")
if DEBUG_MODE and RUN_MODE != "Full":
    raise ValueError("Debug Mode는 RUN_MODE=Full에서만 사용할 수 있습니다.")
if RUN_MODE not in {"Execute", "Audit"} and not DEBUG_MODE and not PROMPT.strip():
    raise ValueError("선택한 실행 모드에서는 PROMPT를 입력해야 합니다.")

performance_mode = str(PERFORMANCE_MODE).strip().lower()
if performance_mode not in {"auto", "latency", "throughput"}:
    raise ValueError("PERFORMANCE_MODE은 Auto, Latency, Throughput 중 하나여야 합니다.")
os.environ["MMM_PERFORMANCE_MODE"] = performance_mode
os.environ["MMM_REMOTE_TRAJECTORY_STORE_CONSENT"] = "1" if ALLOW_REMOTE_TRAJECTORY_STORE else "0"
os.environ["MMM_AGENT_TOOLS"] = "1"
if REFERENCE_MOD_URLS.strip():
    os.environ["MMM_REFERENCE_MOD_URLS"] = REFERENCE_MOD_URLS.strip()
else:
    os.environ.pop("MMM_REFERENCE_MOD_URLS", None)


In [ ]:
# @title 2. GitHub 최신 main 설치
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/M.M.M-Make-Mincraft-Mode")
REPO_URL = "https://github.com/jujumelona/M.M.M-Make-Mincraft-Mode.git"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "+main:refs/remotes/origin/main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "refs/remotes/origin/main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Git 저장소가 아닌 경로가 이미 있습니다: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

USED_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("GitHub commit:", USED_COMMIT)

if RUN_MODE == "Audit":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]", "build"], check=True)
    OUTPUT_ROOT = str(REPO_DIR / "audit")
else:
    setup_script = REPO_DIR / "tools" / "colab_runtime_setup.py"
    spec = importlib.util.spec_from_file_location("_mmm_colab_runtime_setup", setup_script)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load Colab setup script: {setup_script}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    state = module.setup_colab_runtime(
        repo_dir=REPO_DIR,
        used_commit=USED_COMMIT,
        model_profile=MODEL_PROFILE,
        save_to_google_drive=SAVE_TO_GOOGLE_DRIVE,
        remote_base_url=REMOTE_BASE_URL,
        remote_text_model=REMOTE_TEXT_MODEL,
        remote_image_model=REMOTE_IMAGE_MODEL,
        remote_speech_model=REMOTE_SPEECH_MODEL,
        transformers_was_loaded="transformers" in sys.modules,
        engine_was_loaded=any(n == "minecraft_mod_ai" or n.startswith("minecraft_mod_ai.") for n in sys.modules),
        engine_module_file=getattr(sys.modules.get("minecraft_mod_ai"), "__file__", "") or "",
        previous_commit="",
    )
    OUTPUT_ROOT = state["output_root"]


In [ ]:
# @title 3. 기존 모드 입력
if RUN_MODE == "Audit":
    EXISTING_INPUT = None
else:
    from minecraft_mod_ai.colab_run_modes import prepare_existing_mod_input
    EXISTING_INPUT = prepare_existing_mod_input(RUN_MODE)


In [ ]:
# @title 4. 설치 확인
if RUN_MODE == "Audit":
    print("Audit 준비 완료")
else:
    from minecraft_mod_ai import ModelRegistry
    registry = ModelRegistry().to_public_dict()
    if MODEL_PROFILE not in registry["profiles"]:
        raise ValueError(f"지원하지 않는 모델 프로필: {MODEL_PROFILE}")
    print("모델 프로필:", MODEL_PROFILE)
    print("Debug Mode:", "ON — planner skip / example plan injection" if DEBUG_MODE else "OFF")
    print("결과 저장 위치:", OUTPUT_ROOT)


In [ ]:
# @title 5. 플랜 준비
import os
from pathlib import Path
import traceback

os.environ["MMM_LLAMA_KV_AUTOTUNE"] = "1" if KV_CACHE_AUTOTUNE else "0"

session = reply = FINAL_PLAN_PATH = None
PLAN_ERROR = PLAN_ERROR_PATH = None
if RUN_MODE != "Audit":
    from minecraft_mod_ai import CompleteModAISession
    from minecraft_mod_ai.colab_run_modes import resolve_plan_path, run_plan_dialog
    try:
        session = CompleteModAISession(
            output_root=OUTPUT_ROOT,
            minecraft_version=MINECRAFT_VERSION,
            loader=MOD_LOADER,
            model_profile=MODEL_PROFILE,
            existing_input=EXISTING_INPUT,
            fast_mode=False,
            kv_cache_quant=KV_CACHE_QUANT,
        )
        plan_path = resolve_plan_path(run_mode=RUN_MODE, output_root=OUTPUT_ROOT, configured_path=PLAN_FILE)
        dialog = run_plan_dialog(
            session=session,
            run_mode=RUN_MODE,
            prompt=PROMPT,
            plan_path=plan_path,
            debug_mode=DEBUG_MODE,
            minecraft_version=MINECRAFT_VERSION,
            loader=MOD_LOADER,
        )
        reply = dialog.reply
        FINAL_PLAN_PATH = dialog.plan_path
        print("플랜 준비 완료:", FINAL_PLAN_PATH)
        if DEBUG_MODE:
            print("Debug Mode: planner=SKIPPED, implementation=ENABLED")
    except Exception as exc:
        PLAN_ERROR = exc
        PLAN_ERROR_PATH = Path(OUTPUT_ROOT) / "planner_failure.log"
        PLAN_ERROR_PATH.parent.mkdir(parents=True, exist_ok=True)
        PLAN_ERROR_PATH.write_text(traceback.format_exc(), encoding="utf-8")
        traceback.print_exc()


In [ ]:
# @title 6. 제작 또는 Audit
import json
import subprocess
import sys

BUILD_RESULT = None
AUDIT_REPORT_PATH = AUDIT_LOG_PATH = None
if RUN_MODE == "Audit":
    audit_script = REPO_DIR / "tools" / "full_project_audit.py"
    subprocess.run([sys.executable, str(audit_script)], cwd=REPO_DIR, check=False)
    AUDIT_REPORT_PATH = REPO_DIR / "audit" / "FULL_PROJECT_AUDIT.json"
    AUDIT_LOG_PATH = REPO_DIR / "audit" / "FULL_PROJECT_AUDIT.log"
    if not AUDIT_REPORT_PATH.is_file():
        raise FileNotFoundError(AUDIT_REPORT_PATH)
    report = json.loads(AUDIT_REPORT_PATH.read_text(encoding="utf-8"))
    print("Audit overall:", report.get("overall_status"))
    print("실패 검사:", report.get("failed_checks", []))
else:
    from minecraft_mod_ai import CompleteExecutionOptions
    from minecraft_mod_ai.colab_run_modes import should_build
    if not should_build(RUN_MODE):
        print("Plan: 제작 생략")
    elif PLAN_ERROR is not None:
        print("플랜 실패로 제작 보류:", PLAN_ERROR_PATH)
    else:
        options = CompleteExecutionOptions(
            run_blockbench=RUN_BLOCKBENCH,
            run_runtime=RUN_RUNTIME,
            run_client=RUN_CLIENT,
            run_mineflayer=RUN_MINEFLAYER,
            run_visual_review=RUN_VISUAL_REVIEW,
            eula_accepted=ACCEPT_EULA,
            server_launcher=SERVER_LAUNCHER or None,
            screenshot_paths=tuple(SCREENSHOTS),
            resume=True,
        )
        BUILD_RESULT = session.build(reply, run_name=RUN_NAME, options=options)
        print("제작 상태:", BUILD_RESULT.status)
        print("결과 ZIP:", BUILD_RESULT.release_zip)


In [ ]:
# @title 7. 결과 다운로드
from pathlib import Path
import zipfile

target = None
if RUN_MODE == "Audit":
    target = Path("/content/mmm-audit-report.zip")
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in (AUDIT_REPORT_PATH, AUDIT_LOG_PATH):
            if p is not None and Path(p).is_file():
                z.write(p, arcname=Path(p).name)
elif RUN_MODE == "Plan" and FINAL_PLAN_PATH:
    target = Path(FINAL_PLAN_PATH)
elif BUILD_RESULT is not None and BUILD_RESULT.release_zip:
    target = Path(BUILD_RESULT.release_zip)

if target is None or not target.is_file():
    print("다운로드할 결과가 없습니다.")
else:
    print("결과:", target)
    try:
        from google.colab import files as colab_files
        colab_files.download(str(target))
    except ImportError:
        print(target.resolve())


## Debug Mode 계약

`DEBUG_MODE=True`는 **Full에서만** 허용됩니다. planner를 호출하지 않고 host-owned 예제 `CompleteProposal`을 생성·검증하여 `session.load_plan()` 후 일반 `session.build()`로 들어갑니다. 기존 내부 `fast_mode` 플래그는 항상 `False`로 두므로 Debug Mode가 코더 context/품질 예산을 축소하지 않습니다. 기존 프로젝트 전체 진단 기능은 **Audit**으로 분리했습니다.
